# Séance 4 : Introduction aux grands modèles de langage (LLM) avec Ollama
## Construire un mini RAG sur le corpus de presse olympique

**Durée :** 1 journée (7h)
**Public :** Chercheurs et chercheuses en SHS ayant suivi les Séances 1 à 3 (pandas, fouille de texte, réseaux)
**Données :** `olympic_corpus.csv` (le corpus de presse utilisé en Séances 2 et 3)

---

## Déroulé de la journée (indicatif, à ajuster selon le rythme du groupe)

| Horaire | Durée | Bloc | Contenu |
|---|---|---|---|
| 9h00 – 9h15 | 15 min | 0. Introduction | Qu'est-ce qu'un LLM ? Pourquoi des modèles locaux pour la recherche ? |
| 9h15 – 10h00 | 45 min | 1. Installer et prendre en main Ollama | Installation, premiers modèles, ligne de commande |
| 10h00 – 11h00 | 60 min | 2. Dialoguer avec un LLM local en Python | Prompting, historique de conversation, hallucinations |
| 11h00 – 11h15 | 15 min | ☕ Pause | |
| 11h15 – 12h30 | 75 min | 3. Comprendre les embeddings | Fondements du RAG, similarité sémantique |
| 12h30 – 13h30 | 60 min | 🍽️ Déjeuner | |
| 13h30 – 15h00 | 90 min | 4. Construire un mini RAG | Découpage, indexation, recherche, génération augmentée |
| 15h00 – 15h15 | 15 min | ☕ Pause | |
| 15h15 – 16h00 | 45 min | 5. Évaluer et fiabiliser son RAG | Prompt engineering, limites, vérification des sources |
| 16h00 – 16h40 | 40 min | 6. Mini-projet de synthèse | Exercice récapitulatif en autonomie |
| 16h40 – 17h00 | 20 min | Aide-mémoire & clôture | Ressources, questions |

> 💡 **Comment utiliser ce notebook** : chaque section alterne explications, démonstrations et exercices **🧪 À vous de jouer**. Faites les exercices avant de regarder la solution, cachée dans un bloc repliable juste en dessous.

> ⚠️ **Installations nécessaires** :
> ```bash
> # 1. Installer Ollama (application, pas un package Python) :
> #    macOS/Windows : télécharger sur https://ollama.com/download
> #    Linux : curl -fsSL https://ollama.com/install.sh | sh
>
> # 2. Télécharger les deux modèles utilisés aujourd'hui :
> ollama pull llama3.2
> ollama pull nomic-embed-text
>
> # 3. Installer les bibliothèques Python :
> pip install ollama pandas numpy scikit-learn matplotlib seaborn
> ```
> 📁 **Données** : placez `olympic_corpus.csv` dans un dossier `data/` à côté de ce notebook (le même fichier que pour les Séances 2 et 3).

> 🖥️ **Ordinateur portable sans carte graphique dédiée ?** Pas de problème : le modèle `llama3.2` (3 milliards de paramètres) tourne correctement sur processeur (CPU) seul. Les réponses seront un peu plus lentes qu'avec un GPU, mais parfaitement utilisables pour cette séance.


# 0. Introduction

## Qu'est-ce qu'un LLM ?

Un **grand modèle de langage** (*Large Language Model*, LLM) est un modèle statistique entraîné sur d'immenses quantités de texte pour prédire, de façon répétée, le mot (ou fragment de mot, appelé **token**) le plus probable étant donné ce qui précède. C'est ce principe simple, appliqué à une échelle massive, qui produit des systèmes capables de rédiger, résumer, traduire ou répondre à des questions avec une fluidité impressionnante — ChatGPT, Claude, Gemini, Llama, Mistral en sont tous des exemples.

<div class="alert alert-danger" role="alert" style="background-color:#f8d7da;padding:10px;border-radius:5px;">
⚠️ <b>À garder en tête toute la journée</b> : un LLM ne "sait" rien au sens où un humain sait quelque chose — il prédit des suites de mots statistiquement plausibles. Il peut donc produire, avec la même fluidité et la même assurance apparente, une information exacte <b>ou une information entièrement inventée</b> (on parle d'<b>hallucination</b>). Pour un usage en recherche historique, cette distinction est cruciale : un LLM n'est jamais, à lui seul, une source fiable. C'est précisément le problème que la méthode vue aujourd'hui — le RAG — permet de limiter, sans l'éliminer complètement.
</div>

## Pourquoi des modèles **locaux**, avec Ollama ?

Jusqu'ici, la plupart d'entre vous avez sans doute utilisé des LLM via une interface web (ChatGPT, Claude.ai...), où le modèle tourne sur les serveurs distants d'une entreprise. **Ollama** permet de faire tourner des modèles ouverts (*open-weight*) directement sur votre propre machine. Pour un usage en recherche, cela présente plusieurs avantages :

| Avantage | Pourquoi c'est important en recherche |
|---|---|
| **Confidentialité** | Vos données (y compris des archives non publiées, sous embargo, ou sensibles) ne quittent jamais votre ordinateur. |
| **Reproductibilité** | Un modèle local téléchargé aujourd'hui restera identique dans 2 ans ; un modèle en ligne peut être mis à jour ou retiré du jour au lendemain, rendant vos résultats impossibles à reproduire. |
| **Coût** | Gratuit et illimité une fois le modèle téléchargé, sans facturation à l'usage. |
| **Contrôle** | Vous savez précisément quel modèle, quelle version, quels paramètres ont produit un résultat donné — une information souvent nécessaire pour documenter sa méthode dans une publication. |
| **Fonctionnement hors ligne** | Utile en mission d'archives, dans des lieux à connexion limitée. |

En contrepartie, un modèle local de taille raisonnable (pour tourner sur un ordinateur portable) est **moins puissant** qu'un modèle propriétaire de pointe type GPT ou Claude — un compromis à garder à l'esprit selon vos usages.

## Qu'est-ce que le RAG ?

Le **RAG** (*Retrieval-Augmented Generation*, génération augmentée par récupération) est une méthode qui combine deux étapes :

1. **Récupération (retrieval)** : face à une question, on recherche dans un corpus de documents (le vôtre !) les passages les plus pertinents.
2. **Génération augmentée** : on transmet ces passages au LLM, avec la consigne de répondre **en s'appuyant uniquement sur eux**, plutôt que sur ses connaissances générales (souvent floues, potentiellement fausses, jamais spécifiques à votre corpus).

C'est la méthode qui permet de "discuter avec ses propres documents" : construire, par exemple, un assistant capable de répondre à des questions sur notre corpus de presse olympique, **avec des réponses ancrées dans les articles réels**, citations à l'appui — plutôt que de laisser le modèle "deviner" à partir de ce qu'il a mémorisé (de façon souvent imprécise) durant son entraînement.

Aujourd'hui, nous allons construire ce système pas à pas :

1. Prendre en main Ollama et dialoguer avec un LLM local.
2. Observer ses limites (hallucinations) sur des questions précises portant sur notre corpus.
3. Comprendre les *embeddings*, la brique technique qui permet la recherche sémantique.
4. Construire un mini système RAG complet sur le corpus olympique.
5. Apprendre à évaluer et fiabiliser ses réponses.


# 1. Installer et prendre en main Ollama

## Installation

Contrairement aux bibliothèques Python que nous avons utilisées jusqu'ici, **Ollama est une application** à installer séparément (comme R Studio ou Anaconda) :

- **macOS / Windows** : téléchargez l'installeur sur [ollama.com/download](https://ollama.com/download) et suivez les instructions.
- **Linux** : `curl -fsSL https://ollama.com/install.sh | sh` dans un terminal.

Une fois installé, Ollama tourne en arrière-plan comme un petit **serveur local**, accessible à l'adresse `http://localhost:11434` — c'est ce serveur que notre code Python va interroger tout au long de la journée.

## Télécharger des modèles

Ollama distingue deux grandes familles de modèles que nous utiliserons aujourd'hui :

- des modèles de **conversation / génération** (*chat models*), pour dialoguer et rédiger ;
- des modèles d'**embeddings** (voir section 3), spécialisés dans la représentation vectorielle du sens d'un texte — le socle du RAG.

Dans un terminal :

```bash
ollama pull llama3.2          # modèle de conversation (~2 Go, tourne bien sur CPU)
ollama pull nomic-embed-text  # modèle d'embeddings (~300 Mo)
```

<div class="alert alert-success" role="alert" style="background-color:#d4edda;padding:10px;border-radius:5px;">
💡 <b>Autres modèles à essayer, selon la puissance de votre machine</b> : si vous disposez de plus de mémoire (16 Go+ de RAM ou un GPU), <code>llama3.1</code> (8 milliards de paramètres) ou <code>qwen2.5</code> offrent des réponses plus abouties, au prix d'un temps de réponse plus long. La liste complète des modèles disponibles est consultable sur <a href="https://ollama.com/library">ollama.com/library</a>.
</div>

## Tester Ollama en ligne de commande

Avant de passer à Python, testons directement dans le terminal :

```bash
ollama run llama3.2
```

Cette commande ouvre une conversation interactive dans le terminal. Essayez de poser une question, puis tapez `/bye` pour quitter.

Pour lister les modèles déjà téléchargés sur votre machine :

```bash
ollama list
```

## Le client Python

Installons la bibliothèque cliente officielle :

```bash
pip install ollama
```


In [ ]:
import ollama

# Vérifions que tout fonctionne en listant les modèles disponibles localement
ollama.list()


## 🧪 À vous de jouer — Exercice 1

1. Dans un terminal, lancez `ollama run llama3.2` et posez-lui une question sur un sujet que vous connaissez bien (votre propre recherche, par exemple). La réponse vous semble-t-elle fiable ? Précise ? Générique ?
2. Toujours en ligne de commande, testez `ollama run llama3.2 "Résume en une phrase ce que sont les Jeux Olympiques."`
3. Dans la cellule Python ci-dessous, utilisez `ollama.list()` pour vérifier que `llama3.2` et `nomic-embed-text` sont bien tous les deux installés.


In [ ]:
# 🧪 Essayez ici




<details>
<summary>▶️ Voir la solution</summary>

```python
# Question 3
models = ollama.list()
model_names = [m["model"] for m in models["models"]]
print(model_names)
assert any("llama3.2" in m for m in model_names), "llama3.2 non trouvé — avez-vous lancé ollama pull llama3.2 ?"
assert any("nomic-embed-text" in m for m in model_names), "nomic-embed-text non trouvé"
```
</details>


# 2. Dialoguer avec un LLM local en Python

## Un premier appel

La fonction `ollama.chat()` est le point d'entrée principal pour dialoguer avec un modèle de conversation. Elle prend en argument le nom du modèle et une liste de **messages** :

In [ ]:
response = ollama.chat(
    model="llama3.2",
    messages=[
        {"role": "user", "content": "En une phrase, qu'est-ce que les Jeux Olympiques ?"}
    ]
)

print(response["message"]["content"])
# print(response) # pour imprimer la réponse complète, y compris les métadonnées


In [ ]:
print(response)

<div class="alert alert-success" role="alert" style="background-color:#d4edda;padding:10px;border-radius:5px;">
<b>Explications</b>

- Chaque message est un dictionnaire avec deux clés : <code>role</code> (qui parle : <code>"user"</code>, <code>"assistant"</code>, ou <code>"system"</code>) et <code>content</code> (le texte du message).
- <code>response["message"]["content"]</code> contient la réponse générée par le modèle. Le reste de l'objet <code>response</code> contient des métadonnées techniques (durée de génération, nombre de tokens...).
</div>

## Le rôle `system` : donner des instructions générales

Un message de rôle `"system"`, placé en tête de la liste, permet de fixer le comportement général du modèle pour toute la conversation — un peu comme une consigne donnée une fois pour toutes :

In [ ]:
response = ollama.chat(
    model="llama3.2",
    messages=[
        {"role": "system", "content": "Tu es un·e assistant·e de recherche pour historien·nes. Réponds de façon concise et précise, en signalant explicitement quand tu n'es pas certain·e d'une information."},
        {"role": "user", "content": "Quels pays ont boycotté les Jeux Olympiques de 1936 à Berlin ?"}
    ]
)

# print(response["message"]["content"])
print(response)


## Maintenir un historique de conversation

Pour dialoguer sur plusieurs tours (le modèle se souvenant des échanges précédents), il suffit de conserver et d'enrichir la liste `messages`, en y ajoutant à chaque tour la question de l'utilisateur **et** la réponse du modèle :

In [ ]:
messages = [
    {"role": "system", "content": "Tu es un·e assistant·e de recherche pour historien·nes."},
]

def chat_turn(user_message, messages, model="llama3.2"):
    messages.append({"role": "user", "content": user_message})
    response = ollama.chat(model=model, messages=messages)
    messages.append({"role": "assistant", "content": response["message"]["content"]})
    return response["message"]["content"], messages

answer, messages = chat_turn("Qui était le président du Comité International Olympique en 1936 ?", messages)
print(answer)


In [ ]:
# Deuxième tour : le modèle a accès à l'historique de la conversation
answer, messages = chat_turn("Et en 1952 ?", messages)
print(answer)


<div class="alert alert-success" role="alert" style="background-color:#d4edda;padding:10px;border-radius:5px;">

💡 <b>Expérimentations</b> 

Comparez les réponses quand vous...

1. ...réiterez la même question plusieurs fois ;
2. ...demandez à l'IA de citer sa (ou ses) source(s) ; 
3. ...changez la langue dans laquelle vous posez la questions. 
</div>

## Paramètres de génération : la température

Le paramètre `temperature` contrôle le degré d'aléatoire de la génération : une température basse (proche de 0) produit des réponses plus déterministes et factuelles ; une température élevée (proche de 1 ou plus) produit des réponses plus variées, voire créatives — au prix d'une fiabilité moindre pour des tâches factuelles.

In [ ]:
response = ollama.chat(
    model="llama3.2",
    messages=[{"role": "user", "content": "Décris en une phrase l'ambiance des Jeux Olympiques."}],
    options={"temperature": 0.1}
)
print("Température basse :", response["message"]["content"])

response = ollama.chat(
    model="llama3.2",
    messages=[{"role": "user", "content": "Décris en une phrase l'ambiance des Jeux Olympiques."}],
    options={"temperature": 1.2}
)
print("\nTempérature élevée :", response["message"]["content"])


<div class="alert alert-success" role="alert" style="background-color:#d4edda;padding:10px;border-radius:5px;">
💡 <b>Pour un usage de recherche</b> (extraction d'information, réponse factuelle, RAG), on privilégiera presque toujours une température <b>basse</b> (0 à 0.3), afin de limiter la variabilité et l'invention. On réserve une température plus élevée à des usages créatifs (brainstorming, reformulation stylistique...).
</div>

## La limite fondamentale : les hallucinations

Posons maintenant une question **très précise**, sur un article particulier de notre corpus — une information que le modèle n'a probablement jamais vue durant son entraînement :

In [ ]:
response = ollama.chat(
    model="llama3.2",
    messages=[{"role": "user", "content": "Dans l'article intitulé 'OLYMPIC GAMES' publié par le South China Morning Post le 11 septembre 1930, combien de jours devait durer la compétition de Los Angeles en 1932 ?"}],
    options={"temperature": 0}
)
print(response["message"]["content"])


<div class="alert alert-danger" role="alert" style="background-color:#f8d7da;padding:10px;border-radius:5px;">
⚠️ <b>Observez la réponse obtenue</b> : le modèle n'a <b>jamais eu accès</b> à cet article précis. Il va pourtant très probablement produire une réponse assurée — soit en refusant poliment (les meilleurs modèles le font parfois), soit, plus problématique, en <b>inventant un chiffre plausible</b>. C'est exactement le problème que le RAG (sections 3 et 4) va nous permettre de limiter : donner au modèle l'article réel, pour qu'il réponde à partir de ce texte plutôt que de "deviner".
</div>

## 🧪 À vous de jouer — Exercice 2

1. Écrivez un message `system` qui demande au modèle de toujours répondre en français, sur un ton neutre et académique, et de signaler explicitement toute incertitude.
2. Testez une conversation à 3 tours sur un sujet de votre choix, en utilisant la fonction `chat_turn()`.
3. Posez au modèle une question très spécifique sur un événement historique peu connu (une date précise, un score, un nom de second plan). La réponse vous semble-t-elle fiable ? Comment le sauriez-vous sans vérifier ?
4. Comparez deux réponses à la même question factuelle avec `temperature=0` puis `temperature=1.5` : la réponse change-t-elle entre deux exécutions à température égale ? Et entre les deux températures ?


In [ ]:
# 🧪 Essayez ici




<details>
<summary>▶️ Voir la solution</summary>

```python
# Question 1-2
messages = [{"role": "system", "content": "Réponds toujours en français, sur un ton neutre et académique. Si tu n'es pas certain·e d'une information, dis-le explicitement plutôt que d'affirmer."}]
answer, messages = chat_turn("Ma première question", messages)
answer, messages = chat_turn("Ma deuxième question, qui rebondit sur la précédente", messages)
answer, messages = chat_turn("Ma troisième question", messages)

# Question 4
r1 = ollama.chat(model="llama3.2", messages=[{"role": "user", "content": "Quelle est la capitale de la Mongolie ?"}], options={"temperature": 0})
r2 = ollama.chat(model="llama3.2", messages=[{"role": "user", "content": "Quelle est la capitale de la Mongolie ?"}], options={"temperature": 0})
r3 = ollama.chat(model="llama3.2", messages=[{"role": "user", "content": "Quelle est la capitale de la Mongolie ?"}], options={"temperature": 1.5})
```
</details>


# 3. Comprendre les embeddings

## Au-delà du comptage de mots

En Séance 2, nous avons compté des mots et des n-grammes (approche dite *bag-of-words* : le texte est réduit à un sac de mots, sans tenir compte du sens). Cette approche a une limite fondamentale : elle ne reconnaît pas que « *athlète* » et « *sportif* », ou « *a remporté la victoire* » et « *a gagné* », sont **sémantiquement proches**, alors qu'ils ne partagent aucun mot en commun.

Les **embeddings** répondent à ce problème : un modèle d'embedding transforme un texte (mot, phrase, paragraphe) en un **vecteur numérique** (une liste de nombres, typiquement quelques centaines de dimensions) de telle sorte que **deux textes de sens proche produisent des vecteurs proches**, indépendamment des mots exacts employés.

## Calculer un embedding avec Ollama

In [ ]:
response = ollama.embed(model="nomic-embed-text", input="The athlete won the gold medal.")

embedding = response["embeddings"][0]
print(f"Dimension du vecteur : {len(embedding)}")
print(f"Premières valeurs : {embedding[:5]}")


<div class="alert alert-success" role="alert" style="background-color:#d4edda;padding:10px;border-radius:5px;">

**Explications** 

- <code>len(embedding)</code> : 786 signifie que nomic-embed-text représente le texte donné comme un point dans un espace à 768 dimensions. Le choix de 768 dimensions vient de l'architecture et de la configuration du modèle. Ce n'est pas lié au fait que la phrase contient 6 mots : une phrase courte comme une phrase longue produira normalement un vecteur de même dimension.
- L'affichage des premières valeurs est surtout utile pour vérifier que l'embedding a bien été généré. Les valeurs n'ont pas de signification humaine simple et isolée. Pour exploiter réellement ces nombres, on compare généralement deux embeddings, souvent avec la similarité cosinus. Plus le score indique une forte similarité, plus les deux vecteurs pointent dans des directions proches, ce qui correspond généralement à une proximité sémantique.

💡 <code>ollama.embed()</code> accepte aussi bien un texte unique (<code>input="..."</code>) qu'une <b>liste</b> de textes (<code>input=["...", "..."]</code>), auquel cas il renvoie une liste de vecteurs — c'est cette version "par lot" (*batch*) que nous utiliserons en section 4 pour indexer efficacement de nombreux passages à la fois.
</div>

## Mesurer la similarité entre deux textes

Pour comparer deux vecteurs, on utilise la **similarité cosinus** : une valeur entre -1 et 1 (en pratique, souvent entre 0 et 1 pour des textes), d'autant plus proche de 1 que les deux textes sont sémantiquement proches.

In [ ]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

sentences = [
    "The athlete won the gold medal.",     # 0
    "The sportsman achieved victory.",     # 1 — proche du sens de la phrase 0, mots différents
    "It was raining heavily in Shanghai.", # 2 — sujet complètement différent
]

response = ollama.embed(model="nomic-embed-text", input=sentences)
vectors = np.array(response["embeddings"])

similarity_matrix = cosine_similarity(vectors)
similarity_matrix


In [ ]:
import pandas as pd

sim_df = pd.DataFrame(similarity_matrix, index=sentences, columns=[f"phrase {i}" for i in range(len(sentences))])
sim_df


<div class="alert alert-success" role="alert" style="background-color:#d4edda;padding:10px;border-radius:5px;">
💡 On s'attend à observer une similarité <b>élevée</b> entre les phrases 0 et 1 (même sens, mots différents), et une similarité <b>plus faible</b> entre l'une ou l'autre et la phrase 2 (sujet différent) — même si, en comptage de mots pur (section 3 de la Séance 2), ces deux phrases n'auraient partagé <i>aucun</i> mot commun avec la phrase 0 !
</div>

## Pourquoi c'est le socle du RAG

L'idée du RAG (section 4) découle directement de ce principe : si l'on calcule l'embedding de **chaque passage** de notre corpus, puis l'embedding de la **question posée** par l'utilisateur, on peut retrouver les passages les plus **sémantiquement proches** de la question — même si la question n'emploie pas exactement les mêmes mots que les articles. C'est une **recherche par le sens**, complémentaire (et souvent supérieure) à la recherche par mot-clé exact vue en Séance 2 (KWIC).

## 🧪 À vous de jouer — Exercice 3

1. Calculez l'embedding de trois phrases de votre choix (par exemple, en lien avec votre propre recherche) et calculez leur matrice de similarité.
2. Écrivez deux phrases qui parlent du **même sujet** avec un vocabulaire **complètement différent** (aucun mot en commun). Leur similarité cosinus est-elle malgré tout élevée ?
3. Écrivez deux phrases qui partagent plusieurs mots, mais qui ont un sens **très différent** (ex. une phrase affirmative et sa négation). Que remarquez-vous sur leur similarité ?


In [ ]:
# 🧪 Essayez ici




<details>
<summary>▶️ Voir des exemples de solution</summary>

```python
# Question 2 (exemple)
s = ["Le gouvernement a annoncé une hausse des impôts.", "Les autorités ont décidé d'augmenter la fiscalité."]
v = np.array(ollama.embed(model="nomic-embed-text", input=s)["embeddings"])
cosine_similarity(v)

# Question 3 (exemple)
s2 = ["Les athlètes chinois ont remporté la compétition.", "Les athlètes chinois n'ont pas remporté la compétition."]
v2 = np.array(ollama.embed(model="nomic-embed-text", input=s2)["embeddings"])
cosine_similarity(v2)
# Remarque attendue : la similarité reste souvent assez élevée malgré la négation — les embeddings
# capturent surtout la proximité thématique, pas toujours les nuances logiques fines. Une limite à garder en tête.
```
</details>


# 4. Construire un mini RAG sur le corpus olympique

Nous avons maintenant toutes les briques nécessaires. Un système RAG se construit en 5 étapes :

1. **Préparer le corpus** : charger et nettoyer les documents.
2. **Découper (chunking)** : diviser chaque document en passages de taille raisonnable.
3. **Indexer** : calculer l'embedding de chaque passage et les stocker.
4. **Récupérer (retrieve)** : face à une question, retrouver les passages les plus pertinents.
5. **Générer (generate)** : transmettre ces passages au LLM pour qu'il rédige une réponse ancrée dans le corpus.

## Étape 1 : préparer le corpus

Comme en Séances 2 et 3, on recharge et on nettoie le corpus. Pour que la démonstration reste rapide en séance (le calcul d'embeddings sur CPU prend un peu de temps), on se concentre sur un **sous-corpus** ciblé — ici, les articles rédactionnels autour des Jeux de Berlin (1935-1936), un épisode riche en débats (boycott, propagande, participation).

In [ ]:
import re

corpus = pd.read_csv("data/olympic_corpus.csv")
corpus = corpus.drop(columns=["Unnamed: 0"])
corpus["Date"] = pd.to_datetime(corpus["Date"], format="%Y%m%d", errors="coerce")
corpus["Year"] = corpus["Date"].dt.year

def clean_text(text):
    if pd.isna(text):
        return ""
    return re.sub(r"\s+", " ", text).strip()

corpus["Text_clean"] = corpus["Text"].apply(clean_text)

editorial_types = ["Feature/Article", "General News", "Editorial/Opinion", "Review"]
articles = corpus[corpus["category_clean"].isin(editorial_types)].copy()
articles["n_words"] = articles["Text_clean"].str.split().str.len()

# Sous-corpus ciblé pour la démonstration (vous pourrez élargir chez vous)
subset = articles[(articles["Year"] >= 1935) & (articles["Year"] <= 1936) & (articles["n_words"] >= 50)].copy()

print(f"{len(subset)} articles dans le sous-corpus")


## Étape 2 : découper les documents en passages (*chunking*)

Un article entier est souvent trop long pour être un bon "passage" à indexer : il peut mélanger plusieurs sujets, et un modèle d'embedding représente moins bien un texte long qu'un texte court et focalisé. On découpe donc chaque article en **chunks** de taille raisonnable, avec un léger **chevauchement** (*overlap*) pour éviter de couper une information importante pile à la frontière entre deux chunks :

In [ ]:
def chunk_text(text, chunk_size=180, overlap=40):
    """Découpe un texte en chunks de `chunk_size` mots, avec `overlap` mots de recouvrement."""
    words = text.split()
    chunks = []
    start = 0
    while start < len(words):
        end = start + chunk_size
        chunk = " ".join(words[start:end])
        chunks.append(chunk)
        if end >= len(words):
            break
        start += chunk_size - overlap
    return chunks

# Exemple sur un seul article
example_chunks = chunk_text(subset["Text_clean"].iloc[0])
print(f"{len(example_chunks)} chunk(s) pour cet article")
print(example_chunks[0][:300])


In [ ]:
rows = []
for _, row in subset.iterrows():
    for i, chunk in enumerate(chunk_text(row["Text_clean"])):
        rows.append({
            "chunk_id": f"{row['DocId']}_{i}",
            "DocId": row["DocId"],
            "Title": row["Title"],
            "Source": row["Source"],
            "Date": row["Date"],
            "text": chunk
        })

chunks_df = pd.DataFrame(rows)
print(f"{len(chunks_df)} chunks au total, pour {subset['DocId'].nunique()} articles")
chunks_df.head()


<div class="alert alert-success" role="alert" style="background-color:#d4edda;padding:10px;border-radius:5px;">
<b>Explications</b>

- <code>chunk_size=180</code> : chaque passage fait environ 180 mots — un compromis raisonnable entre précision (des chunks courts permettent une recherche plus ciblée) et contexte (des chunks trop courts perdent le fil du raisonnement).
- <code>overlap=40</code> : les 40 derniers mots d'un chunk sont répétés au début du suivant, pour éviter qu'une phrase importante ne soit coupée en deux et perde son sens dans chacune des deux moitiés.
- Il n'existe pas de taille "parfaite" universelle : le bon réglage dépend du type de texte et de la question posée — n'hésitez pas à expérimenter avec différentes valeurs.
</div>

## Étape 3 : indexer le corpus (calculer les embeddings)

On calcule l'embedding de chaque chunk. Pour ne pas envoyer des milliers de requêtes une par une, on procède **par lots** (*batches*) :

In [ ]:
def embed_texts(texts, model="nomic-embed-text", batch_size=32):
    all_embeddings = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i + batch_size]
        response = ollama.embed(model=model, input=batch)
        all_embeddings.extend(response["embeddings"])
    return np.array(all_embeddings)

chunk_embeddings = embed_texts(chunks_df["text"].tolist())

print(f"Matrice d'embeddings : {chunk_embeddings.shape}")   # (nombre de chunks, dimension du vecteur)


<div class="alert alert-success" role="alert" style="background-color:#d4edda;padding:10px;border-radius:5px;">
💡 C'est l'étape la plus longue de la journée (quelques minutes selon votre machine et la taille du sous-corpus) — l'occasion d'une petite pause pendant que le calcul tourne. Pour un usage réel sur un corpus volumineux, on utiliserait une véritable base de données vectorielle (<a href="https://www.trychroma.com/">ChromaDB</a>, <a href="https://github.com/facebookresearch/faiss">FAISS</a>...) plutôt qu'un simple tableau NumPy — mais le principe reste rigoureusement identique à ce que nous construisons ici "à la main".
</div>

## Étape 4 : la fonction de récupération (retrieval)

Face à une question, on calcule son embedding, puis on la compare à **tous** les embeddings du corpus indexé pour identifier les *k* passages les plus proches :

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

def retrieve(query, k=5):
    query_embedding = np.array(ollama.embed(model="nomic-embed-text", input=[query])["embeddings"])
    similarities = cosine_similarity(query_embedding, chunk_embeddings)[0]

    top_indices = similarities.argsort()[::-1][:k]
    results = chunks_df.iloc[top_indices].copy()
    results["similarity"] = similarities[top_indices]
    return results

results = retrieve("Which countries discussed boycotting the Berlin Olympics?", k=5)
results[["DocId", "Source", "Date", "similarity", "text"]]


## Étape 5 : générer une réponse augmentée

On construit maintenant un **prompt** combinant la question de l'utilisateur et les passages récupérés, avec des consignes explicites pour que le modèle s'appuie **uniquement** sur ces passages :

In [ ]:
def build_prompt(question, context_chunks):
    context = "\n\n".join(
        f"[Source {i+1} — {row['Source']}, {row['Date'].date() if pd.notna(row['Date']) else '?'}, DocId {row['DocId']}]\n{row['text']}"
        for i, (_, row) in enumerate(context_chunks.iterrows())
    )

    prompt = f"""Tu es un·e assistant·e de recherche pour historien·nes. Réponds à la question UNIQUEMENT à partir des extraits de presse fournis ci-dessous.
Si l'information demandée n'apparaît pas dans les extraits, dis-le explicitement plutôt que d'inventer une réponse.
Cite systématiquement les sources utilisées, par leur numéro entre crochets (ex. [Source 2]).

Extraits :
{context}

Question : {question}

Réponse :"""
    return prompt


def ask(question, k=5, model="llama3.2"):
    context_chunks = retrieve(question, k=k)
    prompt = build_prompt(question, context_chunks)
    response = ollama.chat(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        options={"temperature": 0}
    )
    return response["message"]["content"], context_chunks


In [ ]:
answer, sources = ask("Which countries discussed boycotting the Berlin Olympics, and why?")

print(answer)


In [ ]:
# On peut toujours consulter les sources effectivement utilisées pour vérifier la réponse
sources[["Source", "Date", "DocId", "Title", "similarity"]]


<div class="alert alert-success" role="alert" style="background-color:#d4edda;padding:10px;border-radius:5px;">
💡 Comparez cette réponse à celle obtenue en section 2 sur une question tout aussi précise, <b>sans</b> RAG : la différence de fiabilité (et la présence de citations vérifiables) est le principal bénéfice de la méthode.
</div>

## 🧪 À vous de jouer — Exercice 4

1. Posez trois nouvelles questions à `ask()` sur le sous-corpus 1935-1936 (par exemple, sur la participation chinoise, sur un athlète particulier, sur la couverture par un journal donné).
2. Que se passe-t-il si vous posez une question **sans rapport** avec le corpus (par exemple, sur la cuisine française) ? Le modèle répond-il correctement qu'il ne trouve pas l'information ?
3. Modifiez le paramètre `k` (par exemple `k=2` puis `k=10`) sur une même question : comment la réponse évolue-t-elle ?
4. **Bonus** : construisez un sous-corpus différent (une autre période, un autre mot-clé de filtrage) et reconstruisez l'index (chunks + embeddings) pour ce nouveau sous-corpus.


In [ ]:
# 🧪 Essayez ici




<details>
<summary>▶️ Voir la solution</summary>

```python
# Question 2 (exemple)
answer, sources = ask("Quelle est la meilleure recette de ratatouille ?")
print(answer)
# Le modèle devrait, en principe, indiquer que les extraits fournis ne traitent pas de ce sujet —
# un bon réflexe à vérifier systématiquement, mais qui n'est jamais garanti à 100% (voir section 5).

# Question 3
answer_k2, _ = ask("Quelle a été la position du Comité International Olympique ?", k=2)
answer_k10, _ = ask("Quelle a été la position du Comité International Olympique ?", k=10)
```
</details>


# 5. Évaluer et fiabiliser son RAG

<div class="alert alert-danger" role="alert" style="background-color:#f8d7da;padding:10px;border-radius:5px;">
⚠️ <b>Le RAG réduit le risque d'hallucination, il ne l'élimine pas.</b> Un modèle peut encore mal interpréter un passage, extrapoler au-delà de ce qui est écrit, ou combiner deux informations de façon erronée. Toute réponse produite par le système doit être considérée comme une <b>piste de lecture</b>, à vérifier systématiquement dans les sources citées — jamais comme un résultat définitif à citer tel quel dans un travail de recherche.
</div>

## Vérifier une réponse : revenir aux sources

C'est ici que les compétences de la Séance 2 redeviennent précieuses : pour vérifier une affirmation produite par le RAG, on peut relire intégralement l'article source, ou même construire une concordance (KWIC) pour retrouver le passage exact :

In [ ]:
def show_full_source(doc_id, df=corpus):
    row = df[df["DocId"] == doc_id].iloc[0]
    print(f"DocId : {row['DocId']}")
    print(f"Titre : {row['Title']}")
    print(f"Source : {row['Source']} — {row['Date'].date() if pd.notna(row['Date']) else '?'}")
    print("---")
    print(row["Text_clean"])

# Exemple : vérifier le premier article cité dans la dernière réponse RAG
show_full_source(sources.iloc[0]["DocId"])


## Améliorer le prompt de génération

Quelques bonnes pratiques de *prompt engineering* pour un usage RAG fiable :

| Consigne | Pourquoi |
|---|---|
| « Réponds uniquement à partir des extraits fournis » | Réduit le recours aux connaissances générales (potentiellement fausses ou anachroniques) du modèle |
| « Si l'information n'y figure pas, dis-le explicitement » | Encourage le modèle à admettre l'absence de réponse plutôt qu'à inventer |
| « Cite systématiquement tes sources » | Permet une vérification immédiate |
| Température basse (0 à 0.3) | Réduit la variabilité et l'inventivité, utile pour des tâches factuelles |
| Demander une citation **exacte** entre guillemets en plus du résumé | Facilite grandement la vérification (mais vérifiez que la citation existe réellement dans le texte — un modèle peut aussi "halluciner" une citation !) |

Essayons une version enrichie du prompt, demandant une citation exacte :

In [ ]:
def build_prompt_v2(question, context_chunks):
    context = "\n\n".join(
        f"[Source {i+1} — {row['Source']}, {row['Date'].date() if pd.notna(row['Date']) else '?'}, DocId {row['DocId']}]\n{row['text']}"
        for i, (_, row) in enumerate(context_chunks.iterrows())
    )

    prompt = f"""Tu es un·e assistant·e de recherche pour historien·nes, rigoureux·se et prudent·e.

Consignes strictes :
- Réponds UNIQUEMENT à partir des extraits fournis ci-dessous.
- Si l'information n'apparaît pas dans les extraits, réponds : "Cette information n'apparaît pas dans les extraits fournis."
- Pour chaque affirmation, indique la source entre crochets (ex. [Source 2]) ET une courte citation exacte entre guillemets tirée du texte source.
- Ne combine jamais deux sources pour inférer une information qu'aucune des deux ne contient explicitement.

Extraits :
{context}

Question : {question}

Réponse :"""
    return prompt


def ask_v2(question, k=5, model="llama3.2"):
    context_chunks = retrieve(question, k=k)
    prompt = build_prompt_v2(question, context_chunks)
    response = ollama.chat(model=model, messages=[{"role": "user", "content": prompt}], options={"temperature": 0})
    return response["message"]["content"], context_chunks

answer_v2, sources_v2 = ask_v2("Which countries discussed boycotting the Berlin Olympics, and why?")
print(answer_v2)


## Une petite grille d'évaluation manuelle

Pour évaluer sérieusement un système RAG (même artisanal), on peut construire un petit jeu de questions-tests et les évaluer manuellement selon une grille simple :

| Critère | Question à se poser |
|---|---|
| **Pertinence de la récupération** | Les passages récupérés sont-ils réellement liés à la question ? |
| **Fidélité (faithfulness)** | La réponse s'appuie-t-elle uniquement sur les passages fournis, sans extrapoler ? |
| **Exactitude des citations** | Les citations exactes figurent-elles vraiment, mot pour mot, dans le texte source ? |
| **Complétude** | La réponse couvre-t-elle l'ensemble de l'information disponible dans le corpus sur cette question ? |
| **Honnêteté en cas d'absence** | Si l'information n'est pas dans le corpus, le système le signale-t-il clairement ? |


In [ ]:
test_questions = [
    "Which countries discussed boycotting the Berlin Olympics, and why?",
    "What was said about Chinese participation in the 1936 Olympics?",
    "What is the best recipe for a French omelette?",   # question hors-sujet, volontairement
]

for q in test_questions:
    answer, srcs = ask_v2(q, k=4)
    print("QUESTION :", q)
    print("RÉPONSE :", answer)
    print("SOURCES :", srcs["DocId"].tolist())
    print("=" * 80)


## 🧪 À vous de jouer — Exercice 5

1. Pour l'une des réponses obtenues ci-dessus, vérifiez **manuellement**, à l'aide de `show_full_source()`, que la citation exacte donnée par le modèle figure bien mot pour mot dans l'article source.
2. Modifiez `build_prompt_v2` pour demander en plus une réponse en **deux parties** : un résumé en une phrase, puis le détail sourcé. Testez sur une question de votre choix.
3. Construisez votre propre liste de 5 questions-tests sur le sous-corpus 1935-1936, et évaluez chaque réponse selon la grille ci-dessus (un tableau papier ou un petit dataframe suffit).


In [ ]:
# 🧪 Essayez ici




<details>
<summary>▶️ Voir une piste de solution</summary>

```python
# Question 3 (exemple de structure)
evaluation = pd.DataFrame([
    {"question": "...", "pertinence": "oui", "fidelite": "oui", "citations_exactes": "partiellement", "complete": "oui", "honnete_si_absent": "n/a"},
    # ... une ligne par question testée
])
evaluation
```
</details>


# 6. Mini-projet de synthèse

Combinons l'ensemble des méthodes vues aujourd'hui sur une **petite investigation en autonomie** (30-40 min). Travaillez seul·e ou en binôme.

## 🧪 Consignes

1. **Choisissez un sous-corpus** différent de celui utilisé en démonstration (une autre période, un autre mot-clé, un autre journal — vous pouvez réutiliser les filtres des Séances 2 et 3).
2. **Construisez l'index** : découpage en chunks + calcul des embeddings pour ce sous-corpus.
3. **Rédigez 5 questions de recherche** pertinentes pour ce sous-corpus.
4. **Interrogez votre RAG** avec `ask_v2()` sur ces 5 questions.
5. **Évaluez** chaque réponse avec la grille de la section 5, en vérifiant au moins une citation dans le texte source d'origine.
6. **Concluez** en quelques lignes : ce système vous semble-t-il utilisable en l'état pour votre propre recherche ? Quelles précautions faudrait-il prendre ? Quelles améliorations proposeriez-vous (autre modèle, meilleur découpage, plus de contexte...) ?


In [ ]:
# 🧪 Votre code ici — utilisez autant de cellules que nécessaire





<details>
<summary>▶️ Voir un exemple de démarche (sous-corpus : articles mentionnant "China" en 1948-1952)</summary>

```python
# 1-2. Sous-corpus et index
subset2 = articles[
    (articles["Year"] >= 1948) & (articles["Year"] <= 1952) &
    (articles["Text_clean"].str.contains("China", case=False, na=False)) &
    (articles["n_words"] >= 50)
].copy()

rows2 = []
for _, row in subset2.iterrows():
    for i, chunk in enumerate(chunk_text(row["Text_clean"])):
        rows2.append({"chunk_id": f"{row['DocId']}_{i}", "DocId": row["DocId"],
                       "Title": row["Title"], "Source": row["Source"], "Date": row["Date"], "text": chunk})
chunks_df2 = pd.DataFrame(rows2)
chunk_embeddings2 = embed_texts(chunks_df2["text"].tolist())

# 3-4. On adapte retrieve() et ask_v2() pour pointer vers ce nouvel index
def retrieve2(query, k=5):
    query_embedding = np.array(ollama.embed(model="nomic-embed-text", input=[query])["embeddings"])
    similarities = cosine_similarity(query_embedding, chunk_embeddings2)[0]
    top_indices = similarities.argsort()[::-1][:k]
    results = chunks_df2.iloc[top_indices].copy()
    results["similarity"] = similarities[top_indices]
    return results

def ask_v3(question, k=5, model="llama3.2"):
    context_chunks = retrieve2(question, k=k)
    prompt = build_prompt_v2(question, context_chunks)
    response = ollama.chat(model=model, messages=[{"role": "user", "content": prompt}], options={"temperature": 0})
    return response["message"]["content"], context_chunks

for q in ["Did China participate in the 1948 or 1952 Olympics?", "..."]:
    answer, srcs = ask_v3(q)
    print(q, "\n", answer, "\n", "="*80)
```
</details>


# Aide-mémoire

## Glossaire

| Terme | Description |
|---|---|
| **LLM (grand modèle de langage)** | Modèle statistique entraîné à prédire la suite probable d'un texte, à partir d'immenses corpus d'entraînement. |
| **Token** | Unité minimale de traitement d'un LLM (un mot, une partie de mot, un signe de ponctuation). |
| **Fenêtre de contexte (context window)** | Quantité maximale de texte (en tokens) qu'un modèle peut prendre en compte en une seule fois. |
| **Modèle ouvert (open-weight)** | Modèle dont les paramètres entraînés sont publiés et peuvent être exécutés localement (ex. Llama, Mistral, Qwen), par opposition à un modèle propriétaire accessible uniquement via une API. |
| **Quantification (quantization)** | Technique de compression d'un modèle réduisant la précision numérique de ses paramètres, pour qu'il tienne en mémoire sur du matériel grand public. |
| **Température (temperature)** | Paramètre contrôlant le degré d'aléatoire de la génération d'un LLM. |
| **Hallucination** | Production, par un LLM, d'une information fausse ou inventée avec la même fluidité et le même degré d'assurance qu'une information correcte. |
| **Embedding** | Représentation numérique (vecteur) d'un texte, censée capturer son sens ; deux textes de sens proche produisent des vecteurs proches. |
| **Similarité cosinus** | Mesure de proximité entre deux vecteurs, utilisée pour comparer des embeddings. |
| **Chunk / Chunking** | Segment de texte de taille raisonnable, résultat du découpage d'un document plus long ; le chunking est l'opération de découpage. |
| **RAG (Retrieval-Augmented Generation)** | Méthode combinant une étape de recherche dans un corpus (retrieval) et une étape de génération de réponse par un LLM, ancrée dans les passages récupérés. |
| **Prompt** | Le texte d'instruction/question soumis à un LLM. |
| **Prompt engineering** | Pratique consistant à formuler et affiner un prompt pour obtenir de meilleures réponses d'un LLM. |
| **Base de données vectorielle (vector store)** | Système de stockage optimisé pour indexer et rechercher efficacement de grandes quantités d'embeddings (ex. ChromaDB, FAISS, dans un cadre plus avancé que ce que nous avons construit aujourd'hui). |

## Index des fonctions

| Fonction | Bibliothèque | Rôle |
|---|---|---|
| `ollama.list()` | ollama | Liste les modèles installés localement |
| `ollama.chat(model=, messages=, options=)` | ollama | Dialogue avec un modèle de conversation |
| `ollama.embed(model=, input=)` | ollama | Calcule un ou plusieurs embeddings |
| `cosine_similarity()` | scikit-learn | Calcule la similarité cosinus entre vecteurs |
| `np.array()` / `.argsort()` | numpy | Manipulation de tableaux numériques ; tri des indices |
| `str.split()` | Python natif | Découpe un texte en mots (utilisé pour le chunking) |

## Où trouver de l'aide ?

1. **Documentation officielle** : [ollama.com](https://ollama.com), [bibliothèque de modèles](https://ollama.com/library), [dépôt GitHub ollama-python](https://github.com/ollama/ollama-python).
2. **The Programming Historian** : pour des tutoriels sur l'usage critique des LLM en recherche historique ([programminghistorian.org](https://programminghistorian.org/)).
3. **HN Lab Log** (le lab d'Huma-Num) : deux articles de Stéphane Poullyau pour [déveloper un RAG](https://hnlab.huma-num.fr/blog/2025/08/26/explorer-ses-documents-avec-la-RAG/) et son [application](https://hnlab.huma-num.fr/blog/2025/10/13/deployer-la-RAG/) web complète. 
4. Jancovic, Marek. (2026). [RAG for Historians: A Radically New Way of Interacting with Multilingual Collections?](https://www.researchgate.net/publication/403255774_RAG_for_Historians_A_Radically_New_Way_of_Interacting_with_Multilingual_Collections)  
4. **Pour aller plus loin en RAG** : des bibliothèques comme [LangChain](https://python.langchain.com/) ou [LlamaIndex](https://www.llamaindex.ai/) automatisent une grande partie de ce que nous avons construit "à la main" aujourd'hui (chunking, indexation, recherche) — utiles pour des projets plus ambitieux, une fois les principes de base bien compris.
5. **Bases de données vectorielles** : [ChromaDB](https://www.trychroma.com/), [FAISS](https://github.com/facebookresearch/faiss) — pour indexer efficacement des corpus de grande taille (au-delà de ce qu'un simple tableau NumPy peut gérer confortablement).
6. **Stack Overflow, autres LLM (ChatGPT, Claude)** : comme pour tout code Python — donnez le maximum de contexte (message d'erreur complet, modèle utilisé, exemple de données) pour obtenir une réponse pertinente.

<div class="alert alert-danger" role="alert" style="background-color:#f8d7da;padding:10px;border-radius:5px;">
⚠️ <b>Pour conclure</b> : les LLM, y compris augmentés par RAG, restent des outils <b>d'aide à la lecture et à l'exploration</b>, pas des sources. Toute affirmation générée doit être vérifiée dans les documents originaux avant d'être citée dans un travail de recherche — exactement comme on ne citerait jamais un résumé de seconde main sans en vérifier la source primaire.
</div>

<div class="alert alert-success" role="alert" style="background-color:#d4edda;padding:10px;border-radius:5px;">
💡 <b>Ce que ces quatre séances vous ont donné</b> : manipuler des données tabulaires (Séance 1), en extraire du sens à partir du texte brut (Séance 2), en révéler la structure relationnelle (Séance 3), et maintenant dialoguer avec elles de façon interactive et ancrée dans les sources (Séance 4). Ensemble, ces briques forment une boîte à outils computationnelle transposable à la plupart des corpus que vous rencontrerez dans vos propres recherches.
</div>
